# Buổi 25 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `xac_suat.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Dữ liệu, backtest, khoảng từ phần dư (mục 4.1)

In [ ]:
%matplotlib inline
import warnings

import matplotlib.pyplot as plt
import numpy as np
import xac_suat as xs

warnings.simplefilter("ignore")
df = xs.doc_dien()
R = xs.backtest(df)                                   # học lại đầu mỗi tháng, 7/2024 → 12/2025 (khoảng 15 giây)
T = (R["ds"] >= xs.NAM_KIEM).to_numpy()               # chấm trên năm 2025
y = R["y"].to_numpy()[T]
sn = df.set_index("ds")["y"].shift(168).reindex(R["ds"]).to_numpy()[T]
print("giờ chấm:", T.sum(), "| MAE LightGBM:", round(np.abs(y - R["yhat"].to_numpy()[T]).mean()),
      "| MAE seasonal naive:", round(np.nanmean(np.abs(y - sn))))

In [ ]:
Q = xs.quantile_tu_phan_du(R)[T]
print("coverage khoảng 80%:", round(xs.coverage(y, Q[:, 1], Q[:, 7]), 3), "| khoảng 90%:", round(xs.coverage(y, Q[:, 0], Q[:, 8]), 3))
print("tỷ lệ giờ nằm dưới quantile 0,05 … 0,95:", xs.ty_le_duoi(y, Q).round(3).tolist())

## Bước 2 — LightGBM quantile và crossing (mục 4.2)

In [ ]:
Qthô = xs.ma_tran_quantile(R)[T]
print("giờ có crossing (thô):", round(float((np.diff(Qthô, axis=1) < 0).any(axis=1).mean()), 3))
Ql = xs.du_bao_quantile(R)[T]
print("giờ có crossing sau du_bao_quantile:", int((np.diff(Ql, axis=1) < 0).any(axis=1).sum()))
print("tỷ lệ giờ nằm dưới quantile 0,05 … 0,95:", xs.ty_le_duoi(y, Ql).round(3).tolist())

## Bước 3 — Pinball, CRPS, WIS tự viết so với scoringrules (mục 4.3)

In [ ]:
import scoringrules as sr

S = xs.mau_tu_phan_du(R)[T]                           # 200 kịch bản cho mỗi giờ
print("pinball τ = 0,9  tự viết:", round(xs.pinball(y, Ql[:, 7], 0.9), 1), "| scoringrules:",
      round(float(np.mean(sr.quantile_score(y, Ql[:, 7], 0.9))), 1))
print("CRPS từ mẫu      tự viết:", round(xs.crps_mau(y, S), 1), "| scoringrules:", round(float(np.mean(sr.crps_ensemble(y, S))), 1))
tong_pinball = sum(xs.pinball(y, Ql[:, i], t) for i, t in enumerate(xs.MUC)) / 4.5
print("WIS              tự viết:", round(xs.wis(y, Ql), 1), "| tổng pinball ÷ 4,5:", round(tong_pinball, 1))

## Bước 4 — PIT và reliability của bốn mô hình (mục 4.4)

In [ ]:
Rg = xs.backtest(df, tru_muc=False)                   # mô hình học trên số gốc, không trừ mức
bon = {"trong mẫu, giả định chuẩn": xs.quantile_trong_mau_chuan(R)[T],
       "mượn khoảng seasonal naive": xs.quantile_muon_seasonal_naive(R, df)[T],
       "không trừ mức": xs.du_bao_quantile(Rg)[(Rg["ds"] >= xs.NAM_KIEM).to_numpy()],
       "phần dư ngoài mẫu": xs.quantile_tu_phan_du(R)[T]}
fig, axs = plt.subplots(1, 4, figsize=(13, 2.8), sharey=True)
for ax, (ten, Qm) in zip(axs, bon.items(), strict=True):
    mep, h = xs.pit_tu_quantile(y, Qm)
    ax.bar(mep[:-1], h, width=np.diff(mep), align="edge", edgecolor="white")
    ax.axhline(1, ls="--", color="gray")
    ax.set_title(ten, fontsize=9)
plt.show()
for ten, Qm in bon.items():
    print(f"{ten:28s} cov90 {xs.coverage(y, Qm[:, 0], Qm[:, 8]):.3f} | WIS {xs.wis(y, Qm):.0f}")

## Bước 5 — Đuôi: peaks-over-threshold (mục 4.5)

In [ ]:
t = xs.doc_nhiet_do_toi_da()
kq = xs.pot(t, 39.0)
print({k: round(v, 3) for k, v in kq.items()})
for n in (2, 10, 50):
    print(f"mức {n} năm: {xs.muc_lap_lai(kq, n):.1f} °C")

## Bước 6 — Kiểm tra

Trong terminal, thư mục `lab/`: `python lab.py check` — xanh 9/9 là xong.